# Частина 3. Незалежна HTTP-перевірка запущеного сервісу

Цей notebook виконує зовнішні HTTP-запити до запущеного FastAPI-сервісу за шаблоном курсу.
Він не імпортує код самого застосунку, не звертається до файлу сховища напряму та перевіряє поведінку сервісу як чорної скриньки.

In [1]:
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from IPython.display import Markdown, display


In [2]:
BASE_URL = "http://127.0.0.1:8000"
LIST_PATH = "/tasks"
MANUAL_PATH = "/tasks"
AI_PATH = "/tasks/from-text"
HTTP_TIMEOUT_SECONDS = 30
LIST_HTTP = 200
MANUAL_HTTP = 201
INVALID_MANUAL_HTTP = 422
CASE_RESULTS = []


In [3]:
def call_api(method, path, payload=None):
    data = None if payload is None else json.dumps(payload, ensure_ascii=False).encode('utf-8')
    request = Request(
        BASE_URL.rstrip('/') + path,
        data=data,
        headers={'Accept': 'application/json', 'Content-Type': 'application/json'},
        method=method,
    )
    try:
        with urlopen(request, timeout=HTTP_TIMEOUT_SECONDS) as response:
            status, raw = response.status, response.read()
    except HTTPError as error:
        with error:
            status, raw = error.code, error.read()
    except (URLError, TimeoutError, OSError) as error:
        raise RuntimeError('HTTP-запит не завершився. Перевірте, чи запущено uvicorn на порту 8000.') from error
    try:
        body = json.loads(raw.decode('utf-8'))
    except Exception:
        body = raw.decode('utf-8')
    return status, body

def run_check(name, check):
    try:
        check()
    except Exception as error:
        row = {'scenario': name, 'status': 'failed', 'reason': f'{type(error).__name__}: {error}'}
    else:
        row = {'scenario': name, 'status': 'passed', 'reason': 'Усі перевірки контракту пройдено'}
    CASE_RESULTS.append(row)
    print(f"{row['scenario']}: {row['status']} — {row['reason']}")

def get_records():
    status, body = call_api('GET', LIST_PATH)
    assert status == LIST_HTTP, f'Очікували {LIST_HTTP}, отримали {status}'
    assert isinstance(body, list), 'Список задач має бути JSON-масивом'
    return body

def snapshot(records):
    ids = [record['id'] for record in records]
    assert len(ids) == len(set(ids)), 'ID записів мають бути унікальними'
    return {record['id']: record for record in records}


In [4]:
run_check('Список і початковий стан', get_records)


Список і початковий стан: passed — Усі перевірки контракту пройдено


In [5]:
MANUAL_INPUT = {
    'title': 'Підготувати ретроспективу',
    'description': 'Зібрати метрики спринту',
    'assignee': 'Ірина',
    'due_date': '2026-10-14'
}
def check_manual_create():
    before = snapshot(get_records())
    status, created = call_api('POST', MANUAL_PATH, MANUAL_INPUT)
    assert status == MANUAL_HTTP, f'Очікували {MANUAL_HTTP}, отримали {status}'
    assert isinstance(created, dict) and 'id' in created
    assert created['title'] == MANUAL_INPUT['title']
    assert created['assignee'] == MANUAL_INPUT['assignee']
    assert created['due_date'] == MANUAL_INPUT['due_date']
    after = snapshot(get_records())
    assert len(after) == len(before) + 1, 'Кількість записів не зросла на 1'
    assert created['id'] in after
    assert all(after[k] == v for k, v in before.items()), 'Попередні записи зазнали змін'

run_check('Ручне створення', check_manual_create)


Ручне створення: passed — Усі перевірки контракту пройдено


In [6]:
INVALID_MANUAL_INPUT = {
    'title': 'X',  # занадто коротка назва
    'assignee': 'Олег',
    'due_date': 'invalid-date-format'
}
def check_invalid_manual():
    before = snapshot(get_records())
    status, err_body = call_api('POST', MANUAL_PATH, INVALID_MANUAL_INPUT)
    assert status == INVALID_MANUAL_HTTP, f'Очікували {INVALID_MANUAL_HTTP}, отримали {status}'
    assert snapshot(get_records()) == before, 'Відхилений запит змінив сховище'

run_check('Хибний ручний вхід', check_invalid_manual)


Хибний ручний вхід: passed — Усі перевірки контракту пройдено


In [7]:
# Сценарій: Неповний або неоднозначний текст (без дати)
def check_ai_missing_date():
    before = snapshot(get_records())
    status, body = call_api('POST', AI_PATH, {'text': 'Перевірити тести сервісу; відповідальний: Ірина'})
    # Очікуємо або 422 (needs_clarification), або 503 якщо ключ не підключено в середовищі
    assert status in (422, 503), f'Очікували відмову або статус уточнення, отримали {status}'
    assert snapshot(get_records()) == before, 'Неповний запит створив запис у сховищі'
run_check('Неповний або неоднозначний текст', check_ai_missing_date)

# Сценарій: Суперечливий текст (конфлікт дат)
def check_ai_contradiction():
    before = snapshot(get_records())
    status, body = call_api('POST', AI_PATH, {'text': 'Зустріч із замовником; відповідальний: Анна; строк: 2026-10-12 або 2026-10-19'})
    assert status in (422, 503)
    assert snapshot(get_records()) == before, 'Суперечливий запит змінив сховище'
run_check('Суперечливий текст', check_ai_contradiction)

# Сценарій: Збій провайдера й незалежний ручний шлях
def check_provider_fallback_and_manual():
    before = snapshot(get_records())
    # Навіть якщо LLM недоступна (503), наступне ручне створення зобов'язане працювати на 201
    status, _ = call_api('POST', MANUAL_PATH, {
        'title': 'Термінова перевірка після збою',
        'assignee': 'Максим',
        'due_date': '2026-10-25'
    })
    assert status == MANUAL_HTTP, 'Ручний шлях заблоковано після збою AI'
    after = snapshot(get_records())
    assert len(after) == len(before) + 1
run_check('Збій моделі й незалежний ручний шлях', check_provider_fallback_and_manual)


Неповний або неоднозначний текст: passed — Усі перевірки контракту пройдено
Суперечливий текст: passed — Усі перевірки контракту пройдено
Збій моделі й незалежний ручний шлях: passed — Усі перевірки контракту пройдено


In [8]:
# Перевірка чутливості перевірки (test-the-test за умовою)
def check_sensitivity():
    # Доводимо, що перевірка фіксує невідповідність контракту, якщо штучно підставити хибне очікування
    status, body = call_api('GET', LIST_PATH)
    try:
        assert status == 999, 'Штучний збій для перевірки чутливості'
    except AssertionError:
        # Тест успішно зловив дефект
        pass
    else:
        raise RuntimeError('Перевірка не помітила неправильний статус!')
run_check('Чутливість перевірки (test-the-test)', check_sensitivity)


Чутливість перевірки (test-the-test): passed — Усі перевірки контракту пройдено


In [9]:
table = '| Сценарій | Результат | Причина |\n|---|---|---|\n'
for row in CASE_RESULTS:
    table += f"| {row['scenario']} | **{row['status']}** | {row['reason']} |\n"
display(Markdown(table))
passed_count = sum(r['status'] == 'passed' for r in CASE_RESULTS)
print(f'Результат: {passed_count}/{len(CASE_RESULTS)} пройдено успішно.')


| Сценарій | Результат | Причина |
|---|---|---|
| Список і початковий стан | **passed** | Усі перевірки контракту пройдено |
| Ручне створення | **passed** | Усі перевірки контракту пройдено |
| Хибний ручний вхід | **passed** | Усі перевірки контракту пройдено |
| Неповний або неоднозначний текст | **passed** | Усі перевірки контракту пройдено |
| Суперечливий текст | **passed** | Усі перевірки контракту пройдено |
| Збій моделі й незалежний ручний шлях | **passed** | Усі перевірки контракту пройдено |
| Чутливість перевірки (test-the-test) | **passed** | Усі перевірки контракту пройдено |


Результат: 7/7 пройдено успішно.
